# FastAPI — Routing, Path & Query Parameters

Interactive notebook companion to `02_routing_path_query.md`

In [ ]:
from fastapi import FastAPI, Path, Query
from fastapi.testclient import TestClient
from typing import Optional
from enum import Enum

app = FastAPI()
client = TestClient(app)

In [ ]:
# Path parameters
@app.get('/users/{user_id}')
def get_user(user_id: int):
    return {'user_id': user_id, 'type': type(user_id).__name__}

# Valid integer
r = client.get('/users/42')
print('Valid int:', r.json())

# Invalid — string instead of int
r = client.get('/users/abc')
print('Invalid:', r.status_code, r.json()['detail'][0]['msg'])

In [ ]:
# Path parameter with validation
@app.get('/items/{item_id}')
def get_item(item_id: int = Path(..., ge=1, le=1000, description='Item ID (1-1000)')):
    return {'item_id': item_id}

print('Valid:', client.get('/items/500').json())
print('Too small:', client.get('/items/0').status_code)   # 422
print('Too large:', client.get('/items/9999').status_code) # 422

In [ ]:
# Query parameters
@app.get('/products')
def list_products(
    skip: int = 0,
    limit: int = 10,
    search: Optional[str] = None,
    tags: list[str] = Query(default=[])
):
    return {'skip': skip, 'limit': limit, 'search': search, 'tags': tags}

# No params — defaults
print('Defaults:', client.get('/products').json())

# With params
print('With params:', client.get('/products?skip=20&limit=5&search=laptop').json())

# List param
print('Tags:', client.get('/products?tags=python&tags=fastapi').json())

In [ ]:
# Enum path parameter
class Color(str, Enum):
    red   = 'red'
    green = 'green'
    blue  = 'blue'

@app.get('/colors/{color}')
def get_color(color: Color):
    return {'color': color, 'hex': {'red': '#FF0000', 'green': '#00FF00', 'blue': '#0000FF'}[color]}

print('Valid enum:', client.get('/colors/red').json())
print('Invalid enum:', client.get('/colors/purple').status_code)  # 422